In [2]:
%load_ext autoreload
%autoreload 2

import numpy as np
from time import time
import matplotlib.pyplot as plt
from main_semi_sticky_wages_data import ModelClass, constraints

from scipy.optimize import minimize
from scipy.optimize import LinearConstraint

from IPython.display import display, Math

from plots import plot_series

model = ModelClass()

par = model.par
sol = model.sol
sim = model.sim

This model is used to find a certain condition. Currently, I just minimize the change in average wage level, because it was not easy to get by just setting parameters.

In [3]:
optimizer_trials = 1000

eps = 1e-8

A = np.array([
    [0, 0, 0, -1, 1, 0, 0, 0, 0, 0],  # theta_h_y - theta_l_y
    [0, 0, 0, 0, 0, -1, 1, 0, 0, 0]
])

lb = np.array([
    eps,
    eps
])

ub = np.array([
    np.inf,
    np.inf
])

linear_constraint = LinearConstraint(A, lb, ub)

In [12]:
param_spec = {
    "par.phi":   {"value": par.phi,  "bounds": (0.01, 0.99),  "active": True},
    "alpha":    {"value": par.alpha,  "bounds": (0.01, 0.99),  "active": True},
    "mu":     {"value": par.mu,  "bounds": (1.0, 10.0), "active": True},
    "c":        {"value": par.c,  "bounds": (0.1, 100.0),  "active": True},
}


active_names = [k for k, v in param_spec.items() if v["active"]]
x0 = np.array([param_spec[k]["value"] for k in active_names], dtype=float)
bounds = [param_spec[k]["bounds"] for k in active_names]

def apply_params(par, names, values):
    for name, val in zip(names, values):
        setattr(par, name, float(val))
    return par

def objective(x):
    # Initialize all non-active parameters to their default values
    model.setup()

    for name, spec in param_spec.items():
        if not spec["active"]:
            setattr(par, name, spec["value"])

    apply_params(par, active_names, x)

    model.solve(do_print=False)

    if not constraints(par, model.sol, t=0, do_print=False):
        return 1e12

    model.simulate_transition()

    wage_gap_sim = (np.average(model.sim.avg_wage[:, -5::]/model.sim.avg_wage[0, -5::], axis=1) - np.average(model.sim.avg_wage[:, :5]/model.sim.avg_wage[0, :5], axis=1))*100

    wage_gap_data = np.loadtxt('Exogenous_estimation/wage_gap_data_mock.csv', delimiter=',')

    SSE = np.sum((wage_gap_sim - wage_gap_data)**2)

    # print(x, SSE)

    return SSE


def optimize_and_plot(single_run=True, optimizer_trials=optimizer_trials):
    runs = []

    n_runs = 1 if single_run else optimizer_trials

    for i in range(n_runs):
        x0_random = np.array([np.random.uniform(low, high) for (low, high) in bounds], dtype=float)

        result = minimize(
            objective,
            x0=x0_random,
            bounds=bounds,
            method="Nelder-Mead",
        )

        runs.append(result)

        print(result.x, result.fun)

        if not single_run and result.fun < 0.0:
            break

    result = min(runs, key=lambda r: r.fun)

    if not single_run and result.fun >= 0.0:
        print("Warning: Optimization did not converge to a solution with negative objective value.")

    optimized_values = result.x
    apply_params(par, active_names, optimized_values)

    print("Success:", result.success)
    print("Message:", result.message)
    print("Objective value:", result.fun)
    print("Optimized parameters:")
    for name in active_names:
        print("par." + name + " = ", getattr(par, name))

    model.solve()
    model.simulate_transition()

    wage_gap = (
        np.average(model.sim.avg_wage[:, -5::] / model.sim.avg_wage[0, -5::], axis=1)
        - np.average(model.sim.avg_wage[:, :5] / model.sim.avg_wage[0, :5], axis=1)
    ) * 100

    plot_series(wage_gap, title="Wage Gap across Cohorts", ylabel="Wage Gap (%)")

optimize_and_plot(single_run=False, optimizer_trials=20)

[0.23115768 0.83763462 2.33834255 0.1       ] 628.5332833184949


KeyboardInterrupt: 